# SMD FPN Compressor — Colab Sprint

Trains an FPN feature compressor + temporal predictor on Singapore Maritime Dataset.
Runs on T4 GPU (~2-3 hr with shortcuts).

**Cross-account resume:** Checkpoints save to Google Drive. Share the `smd_sprint` folder
with another Google account — they can resume where you left off.
See Cell 13 for instructions.

**Architecture:** ResNet50-FPN (frozen) → Compressor → Quantizer → Decompressor → Entropy → Predictor

**Cells at a glance:**
- Cells 1-4: Setup (run once per Colab session)
- Cells 5-7: Build models + loaders + utilities
- Cells 8-10c: Training phases (run sequentially, each saves checkpoints)
- Cell 11: Resume from any checkpoint
- Cell 12: Final validation + COCO mAP
- Cell 13: Cross-account sharing guide
- Cell 14: Verify saved files


---
## Cell 1: Install Dependencies


In [ ]:
# Cell 1: Install dependencies
import subprocess, sys

deps = ["torch>=2.0", "torchvision>=0.15", "torchmetrics", "pycocotools", "tqdm", "opencv-python", "Pillow", "numpy", "gdown"]
for pkg in deps:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


---
## Cell 2: Mount Google Drive
Checkpoints persist here across accounts.


In [ ]:
# Cell 2: Mount Drive
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/smd_sprint"
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Checkpoints: {DRIVE_DIR}")


---
## Cell 3: Clone Repo


In [ ]:
# Cell 3: Clone repo
import os, sys
REPO_URL = "https://github.com/MAHAMAIA/lewm-vc.git"
if not os.path.exists("/content/lewm-vc"):
    !git clone {REPO_URL} /content/lewm-vc
else:
    %cd /content/lewm-vc
    !git pull

%cd /content/lewm-vc
sys.path.insert(0, "/content/lewm-vc/src")
SPRINT_DIR = "/content/lewm-vc/plans/mi300x-training-sprint"
sys.path.insert(0, SPRINT_DIR)
sys.path = list(dict.fromkeys(sys.path))
for key in list(sys.path_importer_cache):
    if "sprint" in key or "lewm-vc" in key:
        del sys.path_importer_cache[key]
import importlib.util
if importlib.util.find_spec("fpn_backbone") is None:
    from importlib.machinery import SourceFileLoader
    loader = SourceFileLoader("fpn_backbone", f"{SPRINT_DIR}/fpn_backbone.py")
    spec = importlib.util.spec_from_loader("fpn_backbone", loader)
    importlib.util.module_from_spec(spec)
    loader.exec_module(sys.modules["fpn_backbone"])
print("Repo ready")


---
## Cell 4: Download + Extract SMD
One-time, ~15 min. Skip if already downloaded.


In [ ]:
# Cell 4: Download SMD zips
import gdown, requests, time
from pathlib import Path

SMD_DIR = Path("/content/datasets/smd")
SMD_DIR.mkdir(parents=True, exist_ok=True)

def gdrive_download(file_id, output_path, max_retries=3):
    """Download from Google Drive with retry and fallback."""
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 1e6:
        print(f"  Already exists ({output_path.stat().st_size/1e6:.0f} MB), skipping")
        return True
    
    urls = [
        f"https://drive.google.com/uc?id={file_id}&confirm=t",
        f"https://drive.google.com/uc?export=download&id={file_id}",
        f"https://docs.google.com/uc?export=download&id={file_id}",
    ]
    
    for url in urls:
        for attempt in range(max_retries):
            try:
                gdown.download(url, str(output_path), quiet=False, fuzzy=True, use_cookies=False)
                if output_path.exists() and output_path.stat().st_size > 1e6:
                    return True
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(5)
    return False

# ── Source 1: SMD-Plus (VIS Onshore + VIS Onboard, single archive) ──
# Newer upload, better accessibility. Contains both onshore & onboard videos.
SMD_PLUS_ID = "1yokFvx_cJu-Fl5ti1wgF1WPHao_2kcKE"
smd_plus_zip = SMD_DIR / "smd_plus.zip"

print("=" * 60)
print("Downloading SMD-Plus (VIS Onshore + VIS Onboard)...")
print("=" * 60)
ok = gdrive_download(SMD_PLUS_ID, smd_plus_zip)
if not ok:
    print("  SMD-Plus failed. Falling back to individual downloads...")
    # ── Source 2: Individual (fallback) ──
    for fid, fname in [("1HnHyQzhzzDlYh15y9_K1mNZX3grlSDMM", "visible_onshore.zip")]:
        print(f"  Downloading {fname}...")
        gdrive_download(fid, SMD_DIR / fname)

# ── Source 3: NIR (optional, skip if fails) ──
print("\n" + "=" * 60)
print("Downloading NIR On-Shore (optional)...")
print("=" * 60)
nir_zip = SMD_DIR / "nir_onshore.zip"
nir_ok = gdrive_download("13wKWzHqkDQHMHjfuUWjgzwhTrnMoEE8P", nir_zip)
if not nir_ok:
    print("  NIR download failed (non-critical, continuing without it)")

# ── Verify ──
print("\n" + "=" * 60)
print("Download summary:")
for f in sorted(SMD_DIR.glob("*.zip")):
    print(f"  {f.name}: {f.stat().st_size/1e6:.0f} MB")
print("=" * 60)
print("Downloads complete")



In [ ]:
# Cell 4b: Extract frames from SMD videos
import zipfile, cv2, os, subprocess, sys
from tqdm import tqdm
from pathlib import Path

SMD_DIR = Path("/content/datasets/smd")
FRAMES_DIR = SMD_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(["apt-get", "install", "-qq", "unrar"], capture_output=True)
try:
    import rarfile
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rarfile"], capture_output=True)
    import rarfile

def extract_any(archive_path, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if dest_dir.exists() and any(dest_dir.iterdir()):
        return True
    if archive_path.suffix == ".rar":
        with rarfile.RarFile(archive_path) as rf:
            rf.extractall(dest_dir)
        return True
    try:
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest_dir)
        return True
    except zipfile.BadZipFile:
        try:
            with rarfile.RarFile(archive_path) as rf:
                rf.extractall(dest_dir)
            return True
        except:
            return False

video_exts = (".avi", ".mp4", ".mov", ".MOV", ".AVI")

# Step 1: Extract smd_plus.zip → inner zips
smd_plus_dir = SMD_DIR / "smd_plus"
if not smd_plus_dir.exists():
    print("Extracting smd_plus.zip...")
    with zipfile.ZipFile(SMD_DIR / "smd_plus.zip") as zf:
        zf.extractall(smd_plus_dir)

# Step 2: Extract inner zips (VIS_Onboard.zip, VIS_Onshore.zip)
for inner_zip in sorted(smd_plus_dir.rglob("*.zip")):
    if inner_zip.stat().st_size > 1000:
        inner_dir = smd_plus_dir / inner_zip.stem
        if not inner_dir.exists():
            print(f"Extracting {inner_zip.name}...")
            extract_any(inner_zip, inner_dir)

# Step 3: Find all video files
all_videos = []
for ext in video_exts:
    all_videos.extend(smd_plus_dir.rglob(f"*{ext}"))
    all_videos.extend(SMD_DIR.rglob(f"*{ext}"))
all_videos = list(set(all_videos))
print(f"Found {len(all_videos)} video files")

# Step 4: Extract frames
for vpath in tqdm(all_videos, desc="Extracting frames"):
    clip_name = vpath.stem
    clip_dir = FRAMES_DIR / clip_name
    if clip_dir.exists():
        continue
    clip_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(vpath))
    idx = 0
    while True:
        ret, f = cap.read()
        if not ret:
            break
        cv2.imwrite(str(clip_dir / f"frame_{idx+1:04d}.png"), f)
        idx += 1
    cap.release()

n = sum(len(list(d.glob("*.png"))) for d in FRAMES_DIR.iterdir() if d.is_dir()) if FRAMES_DIR.exists() else 0
c = len(list(FRAMES_DIR.iterdir())) if FRAMES_DIR.exists() else 0
print(f"Extracted {n} frames across {c} clips")



---
## Cell 5: Config + Build Models


In [ ]:
# Cell 5: Configuration
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path

SMD_DIR = Path("/content/datasets/smd")
FRAMES_DIR = SMD_DIR / "frames"
FPN_LEVEL = "P4"
LATENT_DIM = 16
HYPER_CHANNELS = 32
PRED_HIDDEN = 128
PRED_LAYERS = 4
PRED_HEADS = 2
CONTEXT_LEN = 4
IMAGE_SIZE = 256
SEQUENCE_LENGTH = 4
INTRA_PERIOD = 4
BATCH_SIZE = 4
LR = 3e-4
MAX_GRAD_NORM = 1.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_PIXELS = (IMAGE_SIZE // 16) ** 2
print(f"Config: {FPN_LEVEL}, latent={LATENT_DIM}, seq={SEQUENCE_LENGTH}, batch={BATCH_SIZE}, device={DEVICE}")


In [ ]:
# Cell 5b: Build models
from lewm_vc.feature_compress import FeatureCompressor, FeatureDecompressor
from lewm_vc.entropy import HyperpriorEntropy
from lewm_vc.quant import Quantizer, QuantMode
from lewm_vc.predictor import LeWMPredictor
from lewm_vc.data import FrameDataset, collate_sequences
from fpn_backbone import ResNet50FPN

backbone = ResNet50FPN().to(DEVICE).eval()
compressor = FeatureCompressor(in_channels=256, latent_dim=LATENT_DIM).to(DEVICE)
decompressor = FeatureDecompressor(latent_dim=LATENT_DIM, out_channels=256).to(DEVICE)
entropy_model = HyperpriorEntropy(latent_dim=LATENT_DIM, hyper_channels=HYPER_CHANNELS).to(DEVICE)
quantizer = Quantizer(mode=QuantMode.TRAINING).to(DEVICE)
predictor = LeWMPredictor(
    latent_dim=LATENT_DIM, hidden_dim=PRED_HIDDEN,
    num_layers=PRED_LAYERS, num_heads=PRED_HEADS, context_len=CONTEXT_LEN,
).to(DEVICE)

def pcount(*ms):
    return sum(p.numel() for m in ms for p in m.parameters() if p.requires_grad)

print(f"Trainable params: {pcount(compressor, decompressor, entropy_model, predictor):,}")


---
## Cell 6: Data Loaders


In [ ]:
# Cell 6: Data loaders
from torch.utils.data import DataLoader
from lewm_vc.data.dataset import FrameDataset, collate_sequences

train_ds = FrameDataset.from_root(FRAMES_DIR, sequence_length=SEQUENCE_LENGTH,
    image_size=IMAGE_SIZE, augment=True, frame_stride=1, split="train")
val_ds = FrameDataset.from_root(FRAMES_DIR, sequence_length=SEQUENCE_LENGTH,
    image_size=IMAGE_SIZE, augment=False, frame_stride=1, split="val")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, collate_fn=collate_sequences, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True, collate_fn=collate_sequences)
print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")


---
## Cell 7: Training Utilities
Run this before any training cell.


In [ ]:
# Cell 7: Training utilities
import time
from tqdm import tqdm

def nll_rate_loss(latents, mu, sigma, step_size):
    half = step_size / 2.0
    prob = torch.special.ndtr((latents + half - mu) / sigma) - torch.special.ndtr((latents - half - mu) / sigma)
    prob = prob.clamp(min=1e-10)
    return (-torch.log(prob) / 0.693147).sum() / latents.shape[0]

def extract_fpn(batch):
    frames = batch["frames"].to(DEVICE)
    B, T = frames.shape[:2]
    pyramid = backbone(frames.view(B * T, *frames.shape[2:]))
    feats = pyramid[FPN_LEVEL]
    return feats.view(B, T, *feats.shape[1:])

@torch.no_grad()
def validate():
    compressor.eval(); decompressor.eval(); entropy_model.eval(); predictor.eval()
    total_mse, total_rate, n = 0.0, 0.0, 0
    ss = quantizer.step_size.item()
    for batch in tqdm(val_loader, desc="Val", leave=False):
        feats_seq = extract_fpn(batch)
        B, T = feats_seq.shape[:2]
        context = []
        for t in range(T):
            feat = feats_seq[:, t]
            latent = compressor(feat)
            if t % INTRA_PERIOD == 0:
                qz = quantizer(latent)
                context.append(qz)
                if len(context) > CONTEXT_LEN:
                    context.pop(0)
                _, p = entropy_model(qz)
                rate = nll_rate_loss(qz, p["mu"], p["sigma"], ss)
                recon = decompressor(qz)
            else:
                pm, _ = predictor.forward(context)
                qr = quantizer(latent - pm)
                recon_latent = pm + qr
                context.append(recon_latent)
                if len(context) > CONTEXT_LEN:
                    context.pop(0)
                _, p = entropy_model(qr)
                rate = nll_rate_loss(qr, p["mu"], p["sigma"], ss)
                recon = decompressor(recon_latent)
            total_mse += F.mse_loss(recon, feat).item() * B
            total_rate += rate.item() * B
            n += B
    avg_mse = total_mse / max(n, 1)
    psnr = 10 * torch.log10(torch.tensor(1.0 / max(avg_mse, 1e-10))).item()
    bpp_val = total_rate / max(n, 1) / N_PIXELS
    compressor.train(); decompressor.train(); entropy_model.train(); predictor.train()
    return {"psnr": psnr, "mse": avg_mse, "bpp": bpp_val}

def save_ckpt(phase, step, opt=None, extra=None):
    ckpt = {"phase": phase, "step": step, "fpn_level": FPN_LEVEL,
        "latent_dim": LATENT_DIM, "hyper_channels": HYPER_CHANNELS,
        "compressor": compressor.state_dict(),
        "decompressor": decompressor.state_dict(),
        "entropy": entropy_model.state_dict(),
        "predictor": predictor.state_dict()}
    if opt is not None:
        ckpt["optimizer"] = opt.state_dict()
    if extra:
        ckpt.update(extra)
    path = f"{DRIVE_DIR}/{phase}_step{step}.pt"
    torch.save(ckpt, path)
    with open(f"{DRIVE_DIR}/{phase}_latest.txt", "w") as f:
        f.write(path)
    print(f"    -> {path}")
    return path

print("Utilities ready")


---
## TRAINING PHASES
Run Cells 8, 9, 10 sequentially. Each saves checkpoints to Drive.

**After Colab disconnect or account switch:**  
1. Restart runtime, re-run Cells 1, 2, 3, 5, 5b, 6, 7  
2. Run Cell 11 to pick the latest checkpoint  
3. Jump to the next uncompleted phase cell


---
## Cell 8: Phase 0 — Autoencoder Warmup
Compressor + decompressor only. ~1 min.


In [ ]:
# Cell 8: Phase 0 — Autoencoder Warmup (3,000 steps)
# Pre-extract FPN features once, then train fast on them.
PHASE0_STEPS = 3000

from tqdm import tqdm

all_feats = []
loader_iter = iter(train_loader)
for i in range(50):
    batch = next(loader_iter)
    x = extract_fpn(batch).reshape(-1, 256, IMAGE_SIZE//16, IMAGE_SIZE//16).cpu()
    all_feats.append(x)
    print(f"  [{i+1}/50] extracted {x.shape[0]} samples", flush=True)
train_feats = torch.cat(all_feats)
print(f"Done: {train_feats.shape[0]} samples", flush=True)

class FeatureDataset(torch.utils.data.Dataset):
    def __init__(self, feats): self.feats = feats
    def __len__(self): return len(self.feats)
    def __getitem__(self, i): return self.feats[i]

fast_loader = DataLoader(FeatureDataset(train_feats), batch_size=16, shuffle=True)

for p in predictor.parameters(): p.requires_grad = False
for p in entropy_model.parameters(): p.requires_grad = False

opt = torch.optim.AdamW([
    {"params": compressor.parameters()},
    {"params": decompressor.parameters()},
], lr=LR, weight_decay=0.01)

train_iter = iter(fast_loader)
t0 = time.time()
for step in range(1, PHASE0_STEPS + 1):
    try:
        x = next(train_iter)
    except StopIteration:
        train_iter = iter(fast_loader)
        x = next(train_iter)
    x = x.to(DEVICE)
    latent = compressor(x)
    recon = decompressor(quantizer(latent))
    loss = F.mse_loss(recon, x)
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(list(compressor.parameters()) + list(decompressor.parameters()), MAX_GRAD_NORM)
    opt.step()
    if step % 100 == 0:
        print(f"step {step}/{PHASE0_STEPS} loss={loss.item():.6f} [{time.time()-t0:.1f}s]", flush=True)
    if step % 1000 == 0:
        save_ckpt("phase0", step, opt=opt)
print(f"Phase 0 done [{time.time()-t0:.0f}s]", flush=True)


In [ ]:
# Cell 9: Phase 1 — Rate-Distortion (15,000 steps)
# Feature normalization + fast pre-extraction
PHASE1_STEPS = 15000

# Compute FPN feature statistics
print("Computing FPN stats...", flush=True)
all_feats = []
loader_iter = iter(train_loader)
for i in range(50):
    batch = next(loader_iter)
    x = extract_fpn(batch).reshape(-1, 256, 16, 16).cpu()
    all_feats.append(x)
    print(f"  [{i+1}/50] extracted {x.shape[0]} samples", flush=True)
all_feats = torch.cat(all_feats)
feat_mean = all_feats.mean(dim=(0,2,3), keepdim=True)
feat_std = all_feats.std(dim=(0,2,3), keepdim=True).clamp(min=1e-6)
print(f"FPN mean: {feat_mean.mean().item():.2f}, std: {feat_std.mean().item():.2f}")

# Pre-extract normalized features
all_norm = []
loader_iter = iter(train_loader)
for i in range(200):
    batch = next(loader_iter)
    x = extract_fpn(batch).reshape(-1, 256, 16, 16).cpu()
    x_norm = (x - feat_mean) / feat_std
    all_norm.append(x_norm)
    print(f"  [{i+1}/200] extracted {x_norm.shape[0]} samples", flush=True)
train_feats = torch.cat(all_norm)
print(f"Normalized features: {train_feats.shape}, mean={train_feats.mean():.3f}, std={train_feats.std():.3f}")

class FeatureDataset(torch.utils.data.Dataset):
    def __init__(self, feats): self.feats = feats
    def __len__(self): return len(self.feats)
    def __getitem__(self, i): return self.feats[i]

fast_loader = DataLoader(FeatureDataset(train_feats), batch_size=16, shuffle=True)

# Phase 1 with normalization wrapper
for p in entropy_model.parameters(): p.requires_grad = True

class NormalizedCompressor(torch.nn.Module):
    def __init__(self, comp, decomp, mean, std):
        super().__init__()
        self.comp = comp; self.decomp = decomp
        self.register_buffer("mean", mean); self.register_buffer("std", std)
    def forward(self, x):
        return self.decomp(self.comp((x - self.mean) / self.std)) * self.std + self.mean

ncomp = NormalizedCompressor(compressor, decompressor, feat_mean.to(DEVICE), feat_std.to(DEVICE))

opt = torch.optim.AdamW([
    {"params": compressor.parameters()},
    {"params": decompressor.parameters()},
    {"params": entropy_model.parameters()},
], lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE1_STEPS)

ss = quantizer.step_size.item()
train_iter = iter(fast_loader)
t0 = time.time()
for step in range(1, PHASE1_STEPS + 1):
    try:
        x = next(train_iter)
    except StopIteration:
        train_iter = iter(fast_loader)
        x = next(train_iter)
    x = x.to(DEVICE)
    
    latent = compressor(x); qz = quantizer(latent)
    recon = ncomp.decomp(qz) * ncomp.std + ncomp.mean
    recon_loss = F.mse_loss(recon, x * ncomp.std + ncomp.mean)
    _, p = entropy_model(qz)
    rate = nll_rate_loss(qz, p["mu"], p["sigma"], ss)
    lam = 5.0 * (1.0 - 0.9 * step / PHASE1_STEPS)
    loss = recon_loss + lam * rate / N_PIXELS
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(list(compressor.parameters()) + list(decompressor.parameters()) + list(entropy_model.parameters()), MAX_GRAD_NORM)
    opt.step(); sched.step()
    
    if step % 1500 == 0:
        psnr = 10 * torch.log10(1.0 / recon_loss.detach()).item()
        bpp = (rate / N_PIXELS).item()
        print(f"step {step}/{PHASE1_STEPS} PSNR={psnr:.2f} BPP={bpp:.4f} [{time.time()-t0:.0f}s]", flush=True)
        print(f"  latent stats: range=[{latent.min().item():.2f}, {latent.max().item():.2f}] std={latent.std().item():.2f}", flush=True)
        save_ckpt("phase1", step, opt=opt)
print(f"Phase 1 done [{time.time()-t0:.0f}s]", flush=True)


---
## Cell 10a: Phase 2 — Pre-extract FPN Features
Run once per dataset. Saves to Drive so Cell 10b can skip extraction.


In [ ]:
# Cell 10a: Pre-extract FPN features + stats, save to Drive
PHASE2_STEPS = 10000
H = W = IMAGE_SIZE // 16
EXTRACT_DIR = os.path.join(DRIVE_DIR, "phase2_pre_extract")
os.makedirs(EXTRACT_DIR, exist_ok=True)
torchsafe = torch.save

paths = {
    "train_seqs": os.path.join(EXTRACT_DIR, "train_seqs.pt"),
    "feat_mean": os.path.join(EXTRACT_DIR, "feat_mean.pt"),
    "feat_std": os.path.join(EXTRACT_DIR, "feat_std.pt"),
}

if all(os.path.exists(p) for p in paths.values()):
    print("Loading pre-extracted features from Drive...")
    train_seqs = torch.load(paths["train_seqs"])
    feat_mean = torch.load(paths["feat_mean"])
    feat_std = torch.load(paths["feat_std"])
else:
    print("Computing FPN stats (50 batches)...", flush=True)
    all_feats = []
    loader_iter = iter(train_loader)
    for i in range(50):
        batch = next(loader_iter)
        x = extract_fpn(batch).reshape(-1, 256, H, W).cpu()
        all_feats.append(x)
        print(f"  [{i+1}/50] extracted {x.shape[0]} samples", flush=True)
    all_feats = torch.cat(all_feats)
    feat_mean = all_feats.mean(dim=(0,2,3), keepdim=True)
    feat_std = all_feats.std(dim=(0,2,3), keepdim=True).clamp(min=1e-6)
    print(f"  mean={feat_mean.mean().item():.2f}, std={feat_std.mean().item():.2f}")

    print("Pre-extracting 200 training batches...", flush=True)
    all_seqs = []
    loader_iter = iter(train_loader)
    for i in range(200):
        batch = next(loader_iter)
        feats_seq = extract_fpn(batch).cpu()
        Bt, Tt = feats_seq.shape[:2]
        flat = feats_seq.reshape(-1, 256, H, W)
        flat = (flat - feat_mean) / feat_std
        all_seqs.append(flat.reshape(Bt, Tt, 256, H, W))
        print(f"  [{i+1}/200] extracted {Bt*Tt} samples", flush=True)
    train_seqs = torch.cat(all_seqs)
    torchsafe(train_seqs, paths["train_seqs"])
    torchsafe(feat_mean, paths["feat_mean"])
    torchsafe(feat_std, paths["feat_std"])
    print(f"Saved {train_seqs.shape[0]} sequences to {EXTRACT_DIR}/")

print(f"Sequences: {train_seqs.shape}, mean={train_seqs.mean():.3f}, std={train_seqs.std():.3f}")


---
## Cell 10b: Phase 2 — Temporal Predictor Training
Loads pre-extracted features from 10a. Re-run after fixes without re-extracting.


In [ ]:
# Cell 10b: Phase 2 — Temporal Predictor Training
H = W = IMAGE_SIZE // 16
EXTRACT_DIR = os.path.join(DRIVE_DIR, "phase2_pre_extract")

train_seqs = torch.load(os.path.join(EXTRACT_DIR, "train_seqs.pt"))
feat_mean = torch.load(os.path.join(EXTRACT_DIR, "feat_mean.pt"))
feat_std = torch.load(os.path.join(EXTRACT_DIR, "feat_std.pt"))
print(f"Loaded {train_seqs.shape[0]} sequences, mean={train_seqs.mean():.3f}")

class FeatureDataset(torch.utils.data.Dataset):
    def __init__(self, seqs): self.seqs = seqs
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i): return self.seqs[i]

fast_loader = DataLoader(FeatureDataset(train_seqs), batch_size=BATCH_SIZE, shuffle=True)

for p in predictor.parameters(): p.requires_grad = True

opt = torch.optim.AdamW([
    {"params": compressor.parameters(), "lr": LR * 0.5},
    {"params": decompressor.parameters(), "lr": LR * 0.5},
    {"params": entropy_model.parameters(), "lr": LR * 0.5},
    {"params": predictor.parameters(), "lr": LR},
], weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE2_STEPS)

ss = quantizer.step_size.item()
train_iter = iter(fast_loader)
t0 = time.time()
# ----- Warmup (500 steps): train compressor+decompressor+entropy only -----
for p in predictor.parameters():
    p.requires_grad = False
warmup_opt = torch.optim.AdamW([
    {"params": compressor.parameters(), "lr": LR * 0.5},
    {"params": decompressor.parameters(), "lr": LR * 0.5},
    {"params": entropy_model.parameters(), "lr": LR * 0.5},
], weight_decay=0.01)
warmup_sched = torch.optim.lr_scheduler.CosineAnnealingLR(warmup_opt, T_max=500)
print("Warmup started (500 steps)...", flush=True)
train_iter_w = iter(fast_loader)
t0_w = time.time()
for step_w in range(1, 501):
    try:
        feats_seq = next(train_iter_w)
    except StopIteration:
        train_iter_w = iter(fast_loader)
        feats_seq = next(train_iter_w)
    feats_seq = feats_seq.to(DEVICE)
    B, T = feats_seq.shape[:2]
    warmup_opt.zero_grad()
    sr_w, srate_w = 0.0, 0.0
    ss_w = quantizer.step_size.item()
    for t in range(T):
        feat = feats_seq[:, t]
        latent = compressor(feat)
        qz = quantizer(latent)
        _, p = entropy_model(qz)
        srate_w = srate_w + nll_rate_loss(qz, p["mu"], p["sigma"], ss_w)
        recon = decompressor(qz)
        sr_w = sr_w + F.mse_loss(recon, feat)
    loss_w = sr_w / T + 1.0 * (srate_w / T) / N_PIXELS
    loss_w.backward()
    nn.utils.clip_grad_norm_([p for m in [compressor, decompressor, entropy_model]
        for p in m.parameters() if p.requires_grad], MAX_GRAD_NORM)
    warmup_opt.step()
    warmup_sched.step()
    if step_w % 100 == 0:
        d_w = sr_w.detach().item() / T
        r_w = srate_w.detach().item() / T
        print(f"  warmup {step_w}/500  d={d_w:.4f}  r={r_w:.4f}  loss={loss_w.item():.4f}  [{time.time()-t0_w:.0f}s]", flush=True)
for p in predictor.parameters():
    p.requires_grad = True
print("Warmup done. Starting Phase 2...", flush=True)
for step in range(1, PHASE2_STEPS + 1):
    try:
        feats_seq = next(train_iter)
    except StopIteration:
        train_iter = iter(fast_loader)
        feats_seq = next(train_iter)
    feats_seq = feats_seq.to(DEVICE)
    lam = 0.5 * (1.0 - 0.5 * step / PHASE2_STEPS)
    B, T = feats_seq.shape[:2]
    opt.zero_grad()
    context = []
    sr, srate, snll = 0.0, 0.0, 0.0
    for t in range(T):
        feat = feats_seq[:, t]
        latent = compressor(feat)
        if t % INTRA_PERIOD == 0:
            qz = quantizer(latent)
            context.append(qz.detach())
            if len(context) > CONTEXT_LEN:
                context.pop(0)
            _, p = entropy_model(qz)
            srate = srate + nll_rate_loss(qz, p["mu"], p["sigma"], ss)
            recon = decompressor(qz)
        else:
            pm, _ = predictor.forward(context)
            snll = snll + predictor.nll_loss(context, latent.detach())
            qr = quantizer(latent - pm)
            recon_latent = pm + qr
            context.append(recon_latent.detach())
            if len(context) > CONTEXT_LEN:
                context.pop(0)
            _, p = entropy_model(qr)
            srate = srate + nll_rate_loss(qr, p["mu"], p["sigma"], ss)
            recon = decompressor(recon_latent)
        sr = sr + F.mse_loss(recon, feat)
    total_loss = (sr / T) + lam * (srate / T) / N_PIXELS + 0.01 * (snll / max(T - 1, 1))
    total_loss.backward()
    nn.utils.clip_grad_norm_([p for m in [compressor, decompressor, entropy_model, predictor]
        for p in m.parameters() if p.requires_grad], MAX_GRAD_NORM)
    opt.step(); sched.step()
    if step % 2000 == 0:
        d = sr.detach().item() / T
        r = srate.detach().item() / T
        n = snll.detach().item() / max(T - 1, 1)
        print(f"  step {step}/{PHASE2_STEPS}  d={d:.4f}  r={r:.4f}  nll={n:.4f}  total={total_loss.item():.4f}  [{time.time()-t0:.0f}s]", flush=True)
        metrics = validate()
        print(f"  [val] PSNR={metrics['psnr']:.2f}  BPP={metrics['bpp']:.4f}", flush=True)
        save_ckpt("phase2", step, opt=opt, extra={"scheduler": sched.state_dict(), "feat_mean": feat_mean, "feat_std": feat_std})
print(f"Phase 2 done [{time.time()-t0:.0f}s]", flush=True)


---
## Cell 10c: Phase 2 — Pre-extract Validation Features
Run once after 10b (or any time). Makes validation 12x faster and frees 3GB system RAM.
Overrides validate() to use cached features. Saves to Drive.


In [ ]:
# Cell 10c: Pre-extract validation features + override validate()
# Run once. Makes validation fast (no FPN backbone) and memory-efficient.
import glob
H = W = IMAGE_SIZE // 16
EXTRACT_DIR = os.path.join(DRIVE_DIR, "phase2_pre_extract")

# Load training stats (needed for normalization)
if 'feat_mean' not in dir() or 'feat_std' not in dir():
    feat_mean = torch.load(os.path.join(EXTRACT_DIR, "feat_mean.pt"))
    feat_std = torch.load(os.path.join(EXTRACT_DIR, "feat_std.pt"))

val_seqs_path = os.path.join(EXTRACT_DIR, "val_seqs.pt")
if os.path.exists(val_seqs_path):
    print("Loading pre-extracted val features from Drive...")
    val_seqs = torch.load(val_seqs_path)
else:
    print("Pre-extracting validation features...", flush=True)
    all_val = []
    for i, batch in enumerate(tqdm(val_loader, desc="Val pre-extract")):
        feats_seq = extract_fpn(batch).cpu()
        Bv, Tv = feats_seq.shape[:2]
        flat = feats_seq.reshape(-1, 256, H, W)
        flat = (flat - feat_mean) / feat_std
        all_val.append(flat.reshape(Bv, Tv, 256, H, W))
    val_seqs = torch.cat(all_val)
    torch.save(val_seqs, val_seqs_path)
    print(f"Saved {val_seqs.shape[0]} val sequences to {val_seqs_path}")
    print(f"Val shape: {val_seqs.shape}, mean={val_seqs.mean():.3f}")

print(f"Val sequences: {val_seqs.shape}, mean={val_seqs.mean():.3f}, std={val_seqs.std():.3f}")

# Override validate() to use pre-extracted features (no FPN backbone)
fs_dev_gpu = feat_std.to(DEVICE)
fm_dev_gpu = feat_mean.to(DEVICE)

@torch.no_grad()
def validate():
    compressor.eval(); decompressor.eval(); entropy_model.eval(); predictor.eval()
    total_mse, total_rate, n = 0.0, 0.0, 0
    ss = quantizer.step_size.item()
    for i in range(0, len(val_seqs), BATCH_SIZE):
        feats_seq = val_seqs[i:i+BATCH_SIZE].to(DEVICE)
        B, T = feats_seq.shape[:2]
        context = []
        for t in range(T):
            feat = feats_seq[:, t]
            latent = compressor(feat)
            if t % INTRA_PERIOD == 0:
                qz = quantizer(latent)
                context.append(qz)
                if len(context) > CONTEXT_LEN:
                    context.pop(0)
                _, p = entropy_model(qz)
                rate = nll_rate_loss(qz, p["mu"], p["sigma"], ss)
                recon = decompressor(qz)
            else:
                pm, _ = predictor.forward(context)
                qr = quantizer(latent - pm)
                recon_latent = pm + qr
                context.append(recon_latent)
                if len(context) > CONTEXT_LEN:
                    context.pop(0)
                _, p = entropy_model(qr)
                rate = nll_rate_loss(qr, p["mu"], p["sigma"], ss)
                recon = decompressor(recon_latent)
            recon_unnorm = recon * fs_dev_gpu + fm_dev_gpu
            feat_unnorm = feat * fs_dev_gpu + fm_dev_gpu
            total_mse += F.mse_loss(recon_unnorm, feat_unnorm).item() * B
            total_rate += rate.item() * B
            n += B
    avg_mse = total_mse / max(n, 1)
    psnr = 10 * torch.log10(torch.tensor(1.0 / max(avg_mse, 1e-10))).item()
    bpp_val = total_rate / max(n, 1) / N_PIXELS
    compressor.train(); decompressor.train(); entropy_model.train(); predictor.train()
    return {"psnr": psnr, "mse": avg_mse, "bpp": bpp_val}

print("Fast validate() ready (pre-extracted val features)")


---
## Cell 11: Resume from Checkpoint
Run 11a first to list, then 11b to load.


In [ ]:
# Cell 11-check: Auto-detect checkpoint directory
# Handles flat path vs nested path from cross-account sharing
import glob, os
candidates = [DRIVE_DIR, os.path.join(DRIVE_DIR, "smd_sprint")]
found = None
for d in candidates:
    ckpts = sorted(glob.glob(os.path.join(d, "phase*_step*.pt")))
    if ckpts:
        found = d
        print(f"Checkpoints found at: {d}")
        break
if found is None:
    print("No checkpoints found in any candidate path.")
    for d in candidates:
        exists = os.path.exists(d)
        print(f"  {d}  exists={exists}")
    print("Edit DRIVE_DIR in Cell 2 and re-run.")
else:
    DRIVE_DIR = found
    print(f"DRIVE_DIR set to: {DRIVE_DIR}")


In [ ]:
# Cell 11a: List checkpoints
import glob
ckpts = sorted(glob.glob(f"{DRIVE_DIR}/phase*_step*.pt"))
if not ckpts:
    print("No checkpoints found in Drive.")
else:
    print("Available checkpoints:")
    for i, c in enumerate(ckpts):
        size_mb = os.path.getsize(c) / 1e6
        print(f"  [{i}] {os.path.basename(c):45s} {size_mb:.1f} MB")


In [ ]:
# Cell 11b: Load checkpoint
# EDIT THIS INDEX to pick which checkpoint from 11a
CKPT_INDEX = 0

ckpts = sorted(glob.glob(f"{DRIVE_DIR}/phase*_step*.pt"))
if not ckpts:
    raise RuntimeError("No checkpoints found")
path = ckpts[CKPT_INDEX]
print(f"Loading: {path}")
state = torch.load(path, map_location=DEVICE, weights_only=False)

compressor.load_state_dict(state["compressor"])
decompressor.load_state_dict(state["decompressor"])
entropy_model.load_state_dict(state["entropy"])
predictor.load_state_dict(state["predictor"])

print(f"Loaded {state['phase']} from step {state.get('step', '?')}")
print("Now run the next uncompleted phase cell (8, 9, or 10).")


---
## Cell 12: Final Validation + COCO mAP


In [ ]:
# Cell 12: Final validation
final = validate()
print(f"\n{'=' * 50}")
print("Final Results")
print(f"{'=' * 50}")
print(f"  PSNR: {final['psnr']:.2f} dB")
print(f"  MSE:  {final['mse']:.6f}")
print(f"  BPP:  {final['bpp']:.4f}")
save_ckpt("final", 0)
print(f"\nAll checkpoints in: {DRIVE_DIR}")


In [ ]:
# Cell 12b: COCO mAP (optional, ~20 min)
RUN_MAP = True

if RUN_MAP:
    COCO_DIR = Path("/content/datasets/coco")
    if not (COCO_DIR / "val2017").exists():
        print("Downloading COCO val2017...")
        COCO_DIR.mkdir(parents=True, exist_ok=True)
        !wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip
        !unzip -q /tmp/val2017.zip -d {COCO_DIR}
        !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /tmp/ann.zip
        !unzip -q /tmp/ann.zip -d {COCO_DIR}
        print("COCO ready")
    from validate import run_map
    class Args:
        ckpt = f"{DRIVE_DIR}/final_step0.pt"
        fpn_level = FPN_LEVEL
        coco_dir = str(COCO_DIR)
        num_images = 200
        device = DEVICE
    result = run_map(Args())
    if result:
        print(f"\nmAP delta: {result['delta_pct']:.1f}% - {result['verdict']}")
else:
    print("Skipped")


---
## Cell 13: Cross-Account Sharing Guide

**If you run out of T4 credits in one account:**

1. Account 1 runs Cells 1-10. Checkpoints accumulate at:
   `/content/drive/MyDrive/smd_sprint/`

2. Account 1 shares the `smd_sprint` folder:
   - Go to https://drive.google.com
   - Find `MyDrive/smd_sprint/`
   - Right-click → **Share** → add Account 2's email → **Editor**

3. Account 2 adds shortcut:
   - Accept the share email
   - In Drive, find it under "Shared with me"
   - Right-click → **Organize > Add shortcut** → choose `MyDrive`
   - Now accessible at `/content/drive/MyDrive/smd_sprint/`

4. Account 2 in Colab:
   - File → Open notebook → GitHub URL
   - Runtime → Change runtime type → T4 GPU
   - Run Cells 1, 2, 3, 5, 5b, 6, 7 (mounts their Drive, shared folder visible)
   - Run Cell 11a + 11b to pick latest checkpoint
   - Run the next phase cell

Checkpoints contain model weights + optimizer state.


---
## Cell 14: Verify Saved Files


In [ ]:
# Cell 14: List checkpoint files
print(f"Files in {DRIVE_DIR}:")
for f in sorted(os.listdir(DRIVE_DIR)):
    size_mb = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1e6
    print(f"  {f:45s} {size_mb:.1f} MB")
